In [2]:
from PreRun import PreRun, PostRun
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import make_scorer
from itertools import product
from datetime import date, datetime
from by_dates_Kfold import k_fold_split_option_a
from tqdm import tqdm
from sklearn.model_selection import GridSearchCV

In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [53]:
best_systems = [10, 50, 51]
systems_cleaned[systems_cleaned['system_id'].isin(best_systems)]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.12,BSk,12,...,True,True,True,True,True,cis family thin-film,thin_film,PVDAQ General,5893,2007
8,50,NREL x-Si 6,"Golden, CO",7,39.7420,-105.1727,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8455,1995
9,51,NREL x-Si 7,"Golden, CO",7,39.7416,-105.1734,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8032,1995


In [4]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))
names_list = ('inverter', 'meter', 'other')
# 1332 dropped!
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 4902, 4903]

In [5]:
def custom_xgb_obj(preds: np.ndarray, eval_data: xgb.DMatrix):
    y_true = eval_data.get_label()
    # first derivative is (2x) * [0.5 signum(x) + 1.5]
    multiplier = 1.5 + 0.5 * np.sign(preds - y_true)
    grad = 2 * (preds - y_true) * multiplier
    hess = 2 * multiplier * np.ones_like(preds)
    return (grad, hess)


def custom_xgb_eval(preds: np.ndarray, eval_data: xgb.DMatrix):
    y_true = eval_data.get_label()
    metric_name = 'weight_over'
    value = PostRun.custom_error(y_true, preds,a=1,b=2)
    is_higher_better = False
    return (metric_name, value,)

In [ ]:
params_b = {
    'objective': custom_obj,
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbose': -1
}

In [6]:
starter_kit = PreRun(50, './test_results/50-None', None, systems_cleaned)
starter_kit.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
starter_kit.add_weather_features_only()
starter_kit.good_end_days_naive(streak=7)
good_ends = starter_kit.end_days_naive.copy(deep=True)
df = starter_kit.amended_data.copy(deep=True)
df['year'] = df['time'].dt.year
my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
df_train = df.iloc[0:int(len(df)*0.8)]
outer_cv = k_fold_split_option_a(
    df_train=df_train,
    good_ends=good_ends,
    n_splits=1,
    window_size=None,
    front_or_back='back',
    gap_day=True,
    return_type='index'
)
last_train, last_test = outer_cv[0]

In [7]:
X_tt = df_train[my_cols]
y_tt = df_train['energy']
X_test = df_train[my_cols]
y_test = df_train['energy']
xgb_last_tt = xgb.DMatrix(data=X_tt,label=y_tt)
xgb_last_ho = xgb.DMatrix(data=X_test,label=y_test)

In [16]:
X_tt.tail()

,year,hour_sin,hour_cos,day_of_year_sin,day_of_year_cos,last_year,2_days_ago,cloud_cover,global_tilted_irradiance,proportion_daytime
58529,2018,1.000000,6.123234e-17,0.628763,-0.777597,1.308301,1.175016,0.00,84.239639,1.0
58530,2018,0.965926,-2.588190e-01,0.628763,-0.777597,2.715960,2.330695,0.00,293.458344,1.0
58531,2018,0.866025,-5.000000e-01,0.628763,-0.777597,3.688462,3.325220,0.00,523.726440,1.0
58532,2018,0.707107,-7.071068e-01,0.628763,-0.777597,4.426120,3.967925,0.32,714.760376,1.0
58533,2018,0.500000,-8.660254e-01,0.628763,-0.777597,4.637175,4.241940,0.22,855.164673,1.0


In [11]:
xgb_last_tt

In [17]:
params_b = {
    'objective': custom_xgb_obj,
    'num_leaves': 31,
    'eta': 0.1,
    'max_depth': 5,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbosity': 0,
    'subsample': 0.8,
    'eval_metric': custom_xgb_eval,
}

In [ ]:
bst_a = xgb.train(params=params_b, dtrain=xgb_last_tt, num_boost_round=100, evals = [xgb_last_ho,], obj=custom_xgb_obj, maximize=False)


TypeError: cannot unpack non-iterable DMatrix object

## Try scikit-learn variant.

In [27]:
def custom_xgb_reg_obj(y_true: np.ndarray, preds: np.ndarray):
    # first derivative is (2x) * [0.5 signum(x) + 1.5]
    multiplier = 1.5 + 0.5 * np.sign(preds - y_true)
    grad = 2 * (preds - y_true) * multiplier
    hess = 2 * multiplier * np.ones_like(preds)
    return (grad, hess)


def custom_xgb_reg_eval(y_true: np.ndarray, preds: np.ndarray):
    metric_name = 'weight_over'
    value = PostRun.custom_error(y_true, preds,a=1,b=2)
    return value

In [28]:
try_again = xgb.XGBRegressor(n_estimators=100, max_depth=5, max_leaves=31, learning_rate=0.1,
                             objective=custom_xgb_reg_obj, eval_metric = custom_xgb_reg_eval)
try_again.fit(X_tt.values, y_tt, eval_set=[(X_test.values, y_test),])


[0]	validation_0-rmse:2.44360	validation_0-custom_xgb_reg_eval:6.01150
[1]	validation_0-rmse:2.24063	validation_0-custom_xgb_reg_eval:5.06068
[2]	validation_0-rmse:2.06270	validation_0-custom_xgb_reg_eval:4.29778
[3]	validation_0-rmse:1.90650	validation_0-custom_xgb_reg_eval:3.68274
[4]	validation_0-rmse:1.76918	validation_0-custom_xgb_reg_eval:3.18494
[5]	validation_0-rmse:1.65030	validation_0-custom_xgb_reg_eval:2.78674
[6]	validation_0-rmse:1.54581	validation_0-custom_xgb_reg_eval:2.46263
[7]	validation_0-rmse:1.45438	validation_0-custom_xgb_reg_eval:2.19916
[8]	validation_0-rmse:1.37544	validation_0-custom_xgb_reg_eval:1.98678
[9]	validation_0-rmse:1.30694	validation_0-custom_xgb_reg_eval:1.81426
[10]	validation_0-rmse:1.24645	validation_0-custom_xgb_reg_eval:1.67106
[11]	validation_0-rmse:1.19417	validation_0-custom_xgb_reg_eval:1.55432
[12]	validation_0-rmse:1.14906	validation_0-custom_xgb_reg_eval:1.45927
[13]	validation_0-rmse:1.10996	validation_0-custom_xgb_reg_eval:1.38132
[1

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",<function cus...00263D21418A0>
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import l

In [21]:
PostRun.custom_error(y_test, y_pred)

np.float64(0.8562987292718526)

In [22]:
config = try_again.best_iteration
print(config)

AttributeError: `best_iteration` is only defined when early stopping is used.

In [39]:
def xgb_one_layer_d(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, streak_len: int,
                    n_splits_outer: int, sample_spacing: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index',
        sample_spacing=sample_spacing)
    param_grid = {
        'num_leaves': (7, 15, 31),
        'max_depth': (5, 7, 10),
        'learning_rate': (0.1, 0.2),
        'subsample': (0.8, 1.0),
        'colsample_bytree': (0.8, 1.0),
    }
    col_names = [str(param_settings) for param_settings in product(
        param_grid['num_leaves'],
        param_grid['max_depth'],
        param_grid['learning_rate'],
        param_grid['subsample'],
        param_grid['colsample_bytree'])]
    outer_test_results = pd.DataFrame(
        np.zeros((len(outer_cv), len(col_names))),
        columns=col_names
    )
    for i, (train_ind, test_ind) in enumerate(tqdm(outer_cv)):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        for j, param_settings in enumerate(product(param_grid['num_leaves'],
            param_grid['max_depth'],
            param_grid['learning_rate'],
            param_grid['subsample'],
            param_grid['colsample_bytree']
        )):
            xgb_reg = xgb.XGBRegressor(
                n_estimators=100,
                max_depth=param_settings[1],
                max_leaves=param_settings[0],
                learning_rate=param_settings[2],
                subsample=param_settings[3],
                colsample_bytree=param_settings[4],
                objective=custom_xgb_reg_obj,
                eval_metric=custom_xgb_reg_eval,
                verbosity=0,
            )
            xgb_reg.fit(X_tt, y_tt)
            y_pred = xgb_reg.predict(X_ho)
            val_error = PostRun.custom_error(y_pred,y_ho,1,2)
            outer_test_results.at[i, col_names[j]] = val_error
    outer_test_results.to_csv(f'./xgboost_results/{system_id}_{met_or_inv}.csv', index=False)
    return outer_test_results

In [40]:
early_results_50_n = xgb_one_layer_d(
    50, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 7, -1, 5
)

100%|██████████| 294/294 [1:42:05<00:00, 20.84s/it]


In [41]:
early_results_51_n = xgb_one_layer_d(
    51, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 7, -1, 5
)

100%|██████████| 258/258 [1:27:08<00:00, 20.27s/it]


In [ ]:
xgb_results_10_n = xgb_one_layer_d(
    10, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 7, -1, 2
)

In [42]:
early_results_51_n.mean(axis=1)

0      1.949127
1      1.109273
2      0.347301
3      0.493588
4      2.077863
         ...   
253    4.011964
254    2.603594
255    1.952726
256    0.201424
257    1.134723
Length: 258, dtype: float64

In [44]:
per_sys_means = early_results_51_n.mean(axis=0)

In [48]:
per_sys_means.name = 'mean'

In [49]:
per_sys_stdev = early_results_51_n.std(axis=0)
per_sys_stdev.name = 'std'

In [50]:
total = pd.merge(per_sys_means, per_sys_stdev, how = 'inner', left_index=True, right_index=True)

In [51]:
total

,mean,std
"(7, 5, 0.1, 0.8, 0.8)",1.173673,1.035820
"(7, 5, 0.1, 0.8, 1.0)",1.176234,1.058969
"(7, 5, 0.1, 1.0, 0.8)",1.177997,1.038767
"(7, 5, 0.1, 1.0, 1.0)",1.183578,1.069728
"(7, 5, 0.2, 0.8, 0.8)",1.196735,1.131202
...,...,...
"(31, 10, 0.1, 1.0, 1.0)",1.096101,1.040104
"(31, 10, 0.2, 0.8, 0.8)",1.163299,1.142018
"(31, 10, 0.2, 0.8, 1.0)",1.174024,1.172401
"(31, 10, 0.2, 1.0, 0.8)",1.181832,1.259740


In [52]:
total.describe()

,mean,std
count,72.000000,72.000000
mean,1.155438,1.096823
std,0.031226,0.055731
min,1.086789,1.011577
25%,1.136545,1.049822
50%,1.163299,1.093613
75%,1.177997,1.128639
max,1.205494,1.259740


In [ ]:
total['z'] = 